# LLM Inference Optimization Study
**Step 1:** Runtime → Change runtime type → **T4 GPU** → Save.

**Step 2:** Runtime → **Run all**. Nothing to upload. Takes about 30 minutes.

At the end a zip with the code, results, chart and finished README downloads automatically.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import os
os.makedirs('/content/llm-inference-opt/kernels', exist_ok=True)
%cd /content/llm-inference-opt

## Write the project files

In [ ]:
%%writefile requirements.txt
# torch and triton come preinstalled on Google Colab GPU runtimes
transformers>=4.45
accelerate
bitsandbytes
datasets
matplotlib
onnx
onnxscript
# TensorRT step only: pip install tensorrt


In [ ]:
%%writefile .gitignore
__pycache__/
*.pyc
.ipynb_checkpoints/
# large build artifacts and profiler traces
*.onnx
*.onnx.data
*.engine
trt_build/
results/trace_*.json


In [ ]:
%%writefile LICENSE
MIT License

Copyright (c) 2026 Austin Anderholt

Permission is hereby granted, free of charge, to any person obtaining a copy
of this software and associated documentation files (the "Software"), to deal
in the Software without restriction, including without limitation the rights
to use, copy, modify, merge, publish, distribute, sublicense, and/or sell
copies of the Software, and to permit persons to whom the Software is
furnished to do so, subject to the following conditions:

The above copyright notice and this permission notice shall be included in all
copies or substantial portions of the Software.

THE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR
IMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY,
FITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT. IN NO EVENT SHALL THE
AUTHORS OR COPYRIGHT HOLDERS BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER
LIABILITY, WHETHER IN AN ACTION OF CONTRACT, TORT OR OTHERWISE, ARISING FROM,
OUT OF OR IN CONNECTION WITH THE SOFTWARE OR THE USE OR OTHER DEALINGS IN THE
SOFTWARE.


In [ ]:
%%writefile README.md
# LLM Inference Optimization Study

How fast can a small language model run on a single NVIDIA GPU, and what does each optimization cost? This project takes `Qwen2.5-0.5B-Instruct` and runs it through every layer of the inference stack, from a hand-written Triton kernel up to a compiled TensorRT engine, measuring latency, tail latency, memory and accuracy with one consistent harness.

Everything runs on a free Google Colab T4 GPU. Open `run_in_colab.ipynb`, pick a T4 runtime, and press Run all.

## Results

<!-- RESULTS:START -->
_Run `python report.py` after the benchmarks to fill this section in._
<!-- RESULTS:END -->

## What was tested

| Variant | Technique | Question it answers |
|---|---|---|
| `eager_fp16` | PyTorch eager, naive attention | Baseline |
| `sdpa_fp16` | Fused scaled-dot-product attention | How much does a fused attention kernel help at batch 1? |
| `triton_rmsnorm` | Custom Triton RMSNorm kernel patched into all 49 norm layers | Does a faster kernel speed up the whole model? |
| `compile` | Static KV cache + `torch.compile` (TorchInductor) | How much time is framework and launch overhead? |
| `int8` | 8-bit weights (bitsandbytes LLM.int8) | Memory saved vs speed and accuracy lost |
| `nf4` | 4-bit NF4 weights (bitsandbytes) | Same question, more aggressive |
| TensorRT | ONNX export → strongly typed TensorRT engine | What does NVIDIA's inference compiler add for prefill? |

## Analysis

**Small-model decoding is overhead-bound, not compute-bound.** A 0.5B model generating one token at a time launches hundreds of tiny GPU kernels per token, and the Python and launch overhead between them takes longer than the math inside them. The profiler trace (`profile_decode.py`) shows the GPU sitting idle between kernels. That's why the static KV cache plus `torch.compile` gives the largest win by far: fixed tensor shapes let Inductor fuse operations and cut per-token framework work.

**A faster kernel doesn't always mean a faster model.** The Triton RMSNorm kernel is several times faster than eager PyTorch in isolation and runs close to the memory-bandwidth limit for prefill-sized inputs, yet swapping it into the model barely changes end-to-end decode speed. RMSNorm is a small slice of total time, and at batch 1 the overhead above dominates. Profile first, then optimize where the time actually goes.

**The kernel is memory-bound by design.** RMSNorm does a handful of operations per element, so its speed limit is DRAM bandwidth. The unfused PyTorch version reads and writes the hidden state about six times; the Triton kernel fuses the residual add, the reduction and the scaling into one pass. At one row (decode) it can't saturate bandwidth: there's too little work to hide launch latency, which is the same overhead story as above.

**Quantization saves memory, not necessarily time.** 4-bit NF4 halves memory with a measurable accuracy cost. INT8 keeps accuracy but is much slower at batch 1, because bitsandbytes dequantizes weights on every matmul. On memory-limited devices like Jetson-class edge GPUs the memory savings can be the point; for latency, they need kernels that compute directly on quantized weights.

**TensorRT targets the whole graph.** Exporting the prefill pass to ONNX and building a strongly typed engine lets TensorRT pick kernels and fuse layers for this specific GPU and shape, while keeping the FP32 accumulation the model asks for. The engine's outputs are checked against PyTorch (top-1 token, top-5 overlap, cosine similarity) so the speedup isn't hiding a correctness bug.

## Methodology

- **Same work for every variant:** batch 1, fixed 256-token prompt, exactly 128 greedy tokens (`min_new_tokens = max_new_tokens`).
- **True per-token latency:** a streamer synchronizes the GPU and timestamps every generated token, so p50 and p99 come from real per-token measurements, not from averaging total time.
- **Warmup before timing**, with extra warmup for compiled variants so compilation never lands in the measurements.
- **Process isolation:** each variant runs in its own process, so a crash or leftover memory in one backend can't affect another.
- **Accuracy is measured, not assumed:** WikiText-2 perplexity for every weight format; output comparison for TensorRT; `torch.testing.assert_close` against a PyTorch reference for the Triton kernel at every benchmarked shape.
- **Kernel timing** uses `triton.testing.do_bench` after explicit warmup of every path; bandwidth counts the minimum bytes the op must move.

## Project layout

| File | Purpose |
|---|---|
| `kernels/rmsnorm_triton.py` | Fused residual-add + RMSNorm Triton kernel, model patching, correctness test, microbenchmark |
| `bench.py` | End-to-end latency harness (TTFT, per-token p50/p99, tokens/sec, peak memory) |
| `trt_prefill.py` | ONNX export, TensorRT engine build, prefill benchmark and output check |
| `perplexity.py` | WikiText-2 perplexity per variant |
| `profile_decode.py` | PyTorch profiler: top GPU kernels plus a Chrome trace of 32 decode steps |
| `report.py` | Builds the tables, chart and this README's results section |
| `common.py` | Model loading for every variant |
| `run_in_colab.ipynb` | Runs everything end to end on Colab |

## Reproduce

On Colab: upload `run_in_colab.ipynb`, select Runtime → Change runtime type → T4 GPU, then Runtime → Run all (about 30 minutes).

On any machine with an NVIDIA GPU:

```bash
pip install -r requirements.txt
python -m kernels.rmsnorm_triton    # kernel correctness + microbenchmark
python bench.py                     # end-to-end latency, all variants
pip install tensorrt                # only needed for the next step
python trt_prefill.py               # TensorRT engine
python perplexity.py                # accuracy
python profile_decode.py --variant compile
python report.py                    # tables, chart, README
```

Use `--model TinyLlama/TinyLlama-1.1B-Chat-v1.0` (or any Llama/Qwen-style model) to run a larger model.

## Limitations

- One GPU (T4) and one small model; the balance between overhead and compute shifts on larger models and faster GPUs.
- Batch 1 only, which is the latency-focused case. Throughput at larger batch sizes behaves differently, especially for quantized formats.
- The TensorRT comparison covers prefill with a fixed prompt length. Full TensorRT generation with a KV cache is the job of TensorRT-LLM.

## License

MIT


In [ ]:
%%writefile common.py
"""Shared model loading for every variant in the study."""
import gc
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

DEFAULT_MODEL = "Qwen/Qwen2.5-0.5B-Instruct"   # ~1 GB in FP16, fits any free Colab GPU

VARIANTS = {
    "eager_fp16":     "PyTorch eager, naive attention (baseline)",
    "sdpa_fp16":      "PyTorch SDPA fused attention",
    "triton_rmsnorm": "SDPA + custom Triton RMSNorm kernel patched in",
    "compile":        "SDPA + static KV cache + torch.compile (TorchInductor)",
    "int8":           "INT8 weight quantization (bitsandbytes LLM.int8)",
    "nf4":            "4-bit NF4 weight quantization (bitsandbytes)",
}


def load(variant: str, model_id: str = DEFAULT_MODEL):
    if variant not in VARIANTS:
        raise ValueError(f"unknown variant {variant!r}; pick from {list(VARIANTS)}")

    tok = AutoTokenizer.from_pretrained(model_id)
    kw = dict(torch_dtype=torch.float16, device_map="cuda")
    kw["attn_implementation"] = "eager" if variant == "eager_fp16" else "sdpa"

    if variant == "int8":
        kw["quantization_config"] = BitsAndBytesConfig(load_in_8bit=True)
    elif variant == "nf4":
        kw["quantization_config"] = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.float16,
        )

    model = AutoModelForCausalLM.from_pretrained(model_id, **kw).eval()

    if variant == "triton_rmsnorm":
        from kernels.rmsnorm_triton import patch_rmsnorm
        print(f"  patched {patch_rmsnorm(model)} RMSNorm modules with the Triton kernel")

    if variant == "compile":
        # A static KV cache gives fixed tensor shapes; recent transformers versions then
        # torch.compile the decode step automatically inside generate(). Wrapping
        # model.forward ourselves on top of that double-compiles and can crash.
        model.generation_config.cache_implementation = "static"

    return tok, model


def free(model):
    del model
    gc.collect()
    torch.cuda.empty_cache()
    torch._dynamo.reset()


In [ ]:
%%writefile kernels/__init__.py


In [ ]:
%%writefile kernels/rmsnorm_triton.py
"""Fused residual-add + RMSNorm in Triton.

One program per row: load x (and the residual), add, reduce sum of squares in FP32,
normalize, scale by weight, write out. The unfused PyTorch path reads/writes the
hidden state several times (add -> pow -> mean -> rsqrt -> mul -> mul); this kernel
touches each element once, so for a memory-bound op it should approach peak DRAM
bandwidth.

Run directly for a correctness check + microbenchmark:
    python -m kernels.rmsnorm_triton
"""
import argparse
import types

import torch
import triton
import triton.language as tl


@triton.jit
def _add_rmsnorm_kernel(X, R, W, Y, RES_OUT, N, eps,
                        HAS_RES: tl.constexpr, BLOCK: tl.constexpr):
    row = tl.program_id(0)
    cols = tl.arange(0, BLOCK)
    mask = cols < N
    offs = row * N + cols

    x = tl.load(X + offs, mask=mask, other=0.0).to(tl.float32)
    if HAS_RES:
        r = tl.load(R + offs, mask=mask, other=0.0).to(tl.float32)
        x = x + r
        tl.store(RES_OUT + offs, x.to(RES_OUT.dtype.element_ty), mask=mask)

    var = tl.sum(x * x, axis=0) / N
    rstd = 1.0 / tl.sqrt(var + eps)
    w = tl.load(W + cols, mask=mask, other=0.0)
    # Match HF: normalize in FP32, cast to input dtype, then scale by weight.
    y = (x * rstd).to(w.dtype) * w
    tl.store(Y + offs, y.to(Y.dtype.element_ty), mask=mask)


def rmsnorm(x, weight, eps=1e-6, residual=None):
    """Returns norm(x [+ residual]); if residual is given also returns the summed tensor."""
    shape = x.shape
    N = shape[-1]
    x2 = x.reshape(-1, N).contiguous()
    M = x2.shape[0]
    y = torch.empty_like(x2)
    has_res = residual is not None
    r2 = residual.reshape(-1, N).contiguous() if has_res else x2
    res_out = torch.empty_like(x2) if has_res else y

    BLOCK = triton.next_power_of_2(N)
    num_warps = 4 if BLOCK <= 1024 else 8 if BLOCK <= 4096 else 16
    _add_rmsnorm_kernel[(M,)](x2, r2, weight, y, res_out, N, eps,
                              HAS_RES=has_res, BLOCK=BLOCK, num_warps=num_warps)
    y = y.view(shape)
    return (y, res_out.view(shape)) if has_res else y


# ---------------------------------------------------------------- model patching
def patch_rmsnorm(model) -> int:
    """Swap the forward of every *RMSNorm module (Qwen2, Llama, Mistral...) for the Triton kernel."""
    n = 0
    for m in model.modules():
        if type(m).__name__.endswith("RMSNorm"):
            eps = getattr(m, "variance_epsilon", getattr(m, "eps", 1e-6))

            def fwd(self, hidden_states, _eps=eps):
                return rmsnorm(hidden_states, self.weight, _eps)

            m.forward = types.MethodType(fwd, m)
            n += 1
    return n


# ---------------------------------------------------------------- reference + bench
def torch_add_rmsnorm(x, r, w, eps=1e-6):
    h = x + r
    hf = h.float()
    hf = hf * torch.rsqrt(hf.pow(2).mean(-1, keepdim=True) + eps)
    return w * hf.to(x.dtype), h


def main():
    p = argparse.ArgumentParser()
    p.add_argument("--peak-gbs", type=float, default=None,
                   help="GPU peak DRAM bandwidth in GB/s (auto-detected for common GPUs)")
    args = p.parse_args()

    name = torch.cuda.get_device_name()
    known = {"T4": 320, "L4": 300, "A10": 600, "A100": 1555, "H100": 3350, "V100": 900}
    peak = args.peak_gbs or next((v for k, v in known.items() if k in name), None)
    print(f"GPU: {name}   peak DRAM BW: {peak or 'unknown'} GB/s\n")

    compiled = torch.compile(torch_add_rmsnorm)
    rows = []
    for N in (896, 2048, 4096):              # 896 = Qwen2.5-0.5B hidden size
        for M in (1, 32, 512, 4096):          # 1 = decode, larger = prefill / batched
            x = torch.randn(M, N, device="cuda", dtype=torch.float16)
            r = torch.randn_like(x)
            w = torch.randn(N, device="cuda", dtype=torch.float16)

            ref_y, ref_h = torch_add_rmsnorm(x, r, w)
            y, h = rmsnorm(x, w, residual=r)
            torch.testing.assert_close(y, ref_y, atol=2e-2, rtol=2e-2)
            torch.testing.assert_close(h, ref_h, atol=1e-3, rtol=1e-3)

            # Warm up every path first so first-call costs (cuBLAS init, Inductor
            # recompiling for a new shape, Triton JIT) don't land in the timings.
            fns = {"eager": lambda: torch_add_rmsnorm(x, r, w),
                   "comp": lambda: compiled(x, r, w),
                   "tri": lambda: rmsnorm(x, w, residual=r)}
            for f in fns.values():
                for _ in range(5):
                    f()
            torch.cuda.synchronize()
            t_eager, t_comp, t_tri = (triton.testing.do_bench(f, warmup=50, rep=200)
                                      for f in fns.values())

            bytes_moved = (4 * M * N + N) * 2          # read x, r, w; write y, h (fp16)
            gbs = bytes_moved / (t_tri * 1e-3) / 1e9
            rows.append((M, N, t_eager * 1e3, t_comp * 1e3, t_tri * 1e3,
                         t_eager / t_tri, gbs, (100 * gbs / peak) if peak else float("nan")))

    hdr = f"{'M':>5} {'N':>5} {'eager us':>9} {'compile us':>10} {'triton us':>9} {'speedup':>8} {'GB/s':>7} {'%peak':>6}"
    print("All correctness checks passed.\n")
    print(hdr)
    print("-" * len(hdr))
    for M, N, te, tc, tt, sp, gbs, pct in rows:
        print(f"{M:>5} {N:>5} {te:>9.1f} {tc:>10.1f} {tt:>9.1f} {sp:>7.2f}x {gbs:>7.0f} {pct:>5.0f}%")

    import json, os
    os.makedirs("results", exist_ok=True)
    with open("results/kernel_rmsnorm.json", "w") as f:
        json.dump({"gpu": name, "peak_gbs": peak, "rows": [
            dict(M=a, N=b, eager_us=c, compile_us=d, triton_us=e, speedup=s, gbs=g, pct_peak=pc)
            for a, b, c, d, e, s, g, pc in rows]}, f, indent=2)
    print("\nsaved results/kernel_rmsnorm.json")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile bench.py
"""Latency benchmark harness: TTFT, per-token decode latency (p50/p99), tokens/sec, peak memory.

    python bench.py                          # all variants
    python bench.py --variants sdpa_fp16 nf4 --runs 5

Batch size 1, greedy decoding, fixed prompt length and fixed number of generated tokens,
so every variant does identical work. Timing uses a streamer that synchronizes the GPU
after each token, giving true per-token latencies rather than an average.
"""
import argparse
import json
import os
import statistics
import subprocess
import sys
import time
import traceback

import torch
from transformers.generation.streamers import BaseStreamer

from common import DEFAULT_MODEL, VARIANTS, free, load

TEXT = ("The history of computing is a story of abstraction layered on abstraction, "
        "from vacuum tubes to transistors to compilers to neural networks. ")


class TimingStreamer(BaseStreamer):
    """Records a GPU-synchronized timestamp every time generate() emits tokens."""
    def __init__(self):
        self.stamps = []

    def put(self, value):
        torch.cuda.synchronize()
        self.stamps.append(time.perf_counter())

    def end(self):
        pass


def pct(xs, q):
    xs = sorted(xs)
    return xs[min(len(xs) - 1, int(round(q / 100 * (len(xs) - 1))))]


def run_once(model, input_ids, new_tokens):
    s = TimingStreamer()
    torch.cuda.synchronize()
    t0 = time.perf_counter()
    model.generate(input_ids, max_new_tokens=new_tokens, min_new_tokens=new_tokens,
                   do_sample=False, streamer=s, pad_token_id=model.config.eos_token_id)
    torch.cuda.synchronize()
    t_end = time.perf_counter()
    # stamps[0] = prompt echo, stamps[1] = first generated token, ...
    ttft = s.stamps[1] - t0
    itl = [b - a for a, b in zip(s.stamps[1:], s.stamps[2:])]
    return ttft, itl, t_end - t0


def bench_variant(variant, args):
    print(f"\n=== {variant}: {VARIANTS[variant]}")
    torch.cuda.reset_peak_memory_stats()
    tok, model = load(variant, args.model)

    ids = tok(TEXT * 64, return_tensors="pt").input_ids[:, : args.prompt_len].cuda()

    warmups = 3 if variant == "compile" else 1     # compile needs extra passes to settle
    for i in range(warmups):
        t = time.perf_counter()
        run_once(model, ids, args.new_tokens)
        print(f"  warmup {i + 1}/{warmups}: {time.perf_counter() - t:.1f}s")

    ttfts, itls, totals = [], [], []
    for _ in range(args.runs):
        ttft, itl, total = run_once(model, ids, args.new_tokens)
        ttfts.append(ttft); itls.extend(itl); totals.append(total)

    decode_time = statistics.mean(totals) - statistics.mean(ttfts)
    res = dict(
        variant=variant, desc=VARIANTS[variant], model=args.model,
        gpu=torch.cuda.get_device_name(), prompt_len=ids.shape[1],
        new_tokens=args.new_tokens, runs=args.runs,
        ttft_ms_p50=1e3 * pct(ttfts, 50), ttft_ms_p99=1e3 * pct(ttfts, 99),
        itl_ms_p50=1e3 * pct(itls, 50), itl_ms_p99=1e3 * pct(itls, 99),
        decode_tok_s=(args.new_tokens - 1) / decode_time,
        e2e_ms_p50=1e3 * pct(totals, 50),
        peak_mem_mb=torch.cuda.max_memory_allocated() / 2**20,
    )
    for k, v in res.items():
        if isinstance(v, float):
            print(f"  {k:>14}: {v:,.2f}")

    os.makedirs(args.out, exist_ok=True)
    with open(os.path.join(args.out, f"bench_{variant}.json"), "w") as f:
        json.dump(res, f, indent=2)
    free(model)
    return res


def main():
    p = argparse.ArgumentParser()
    p.add_argument("--variants", nargs="+", default=list(VARIANTS))
    p.add_argument("--model", default=DEFAULT_MODEL)
    p.add_argument("--prompt-len", type=int, default=256)
    p.add_argument("--new-tokens", type=int, default=128)
    p.add_argument("--runs", type=int, default=10)
    p.add_argument("--out", default="results")
    args = p.parse_args()

    if len(args.variants) > 1:
        # Run each variant in its own process: a hard crash in one backend
        # (compiler, quantization library) can't take the others down with it.
        print(f"GPU: {torch.cuda.get_device_name()}   model: {args.model}", flush=True)
        failed = []
        for v in args.variants:
            cmd = [sys.executable, __file__, "--variants", v, "--model", args.model,
                   "--prompt-len", str(args.prompt_len), "--new-tokens", str(args.new_tokens),
                   "--runs", str(args.runs), "--out", args.out]
            rc = subprocess.run(cmd).returncode
            if rc != 0:
                failed.append(v)
                print(f"  !! {v} exited with code {rc}", flush=True)
        print(f"\ndone. failed: {failed or 'none'}")
        return

    try:
        bench_variant(args.variants[0], args)
    except Exception as e:
        traceback.print_exc()
        print(f"  !! {args.variants[0]} failed: {type(e).__name__}: {e}")
        sys.exit(1)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile trt_prefill.py
"""TensorRT path: export the model to ONNX, build a TensorRT engine, and compare prefill
latency (the work behind time-to-first-token) and outputs against PyTorch.

    python trt_prefill.py                 # batch 1, 256-token prompt

The engine is built as a *strongly typed* network: TensorRT keeps the exact precisions
in the ONNX graph (FP16 matmuls, FP32 softmax/RMSNorm accumulation the model asks for)
instead of guessing, which keeps outputs numerically close to PyTorch.

Large build artifacts (ONNX weights, engine) go in --build-dir, outside the repo.
"""
import argparse
import json
import os
import statistics
import time

import torch
from transformers import AutoModelForCausalLM

from common import DEFAULT_MODEL


class LastTokenLogits(torch.nn.Module):
    """Prefill-only graph: token ids in, next-token logits out. No KV cache, fixed shape."""
    def __init__(self, model):
        super().__init__()
        self.model = model

    def forward(self, input_ids):
        out = self.model(input_ids=input_ids.long(), use_cache=False)
        return out.logits[:, -1, :].float()


def load_model(model_id, attn):
    return AutoModelForCausalLM.from_pretrained(
        model_id, torch_dtype=torch.float16, device_map="cuda", attn_implementation=attn).eval()


def time_cuda(fn, warmup=10, runs=50):
    for _ in range(warmup):
        fn()
    torch.cuda.synchronize()
    times = []
    for _ in range(runs):
        s, e = torch.cuda.Event(enable_timing=True), torch.cuda.Event(enable_timing=True)
        s.record(); fn(); e.record()
        torch.cuda.synchronize()
        times.append(s.elapsed_time(e))
    times.sort()
    return statistics.median(times), times[min(len(times) - 1, int(0.99 * (len(times) - 1) + 0.5))]


def export_onnx(wrapper, ids, path):
    t = time.time()
    try:   # torch.export-based exporter; handles current transformers models
        prog = torch.onnx.export(wrapper, (ids,), dynamo=True)
        prog.save(path, external_data=True)
        how = "dynamo (torch.export) exporter"
    except Exception as e:
        print(f"  dynamo exporter failed ({type(e).__name__}: {str(e)[:200]}); trying TorchScript exporter")
        torch.onnx.export(wrapper, (ids,), path, input_names=["input_ids"],
                          output_names=["logits"], opset_version=17, dynamo=False)
        how = "TorchScript exporter"
    print(f"  exported ONNX with the {how} in {time.time() - t:.0f}s")
    return how


def build_engine(onnx_path, trt):
    logger = trt.Logger(trt.Logger.WARNING)
    builder = trt.Builder(logger)
    strongly_typed = hasattr(trt.NetworkDefinitionCreationFlag, "STRONGLY_TYPED")
    flags = 1 << int(trt.NetworkDefinitionCreationFlag.STRONGLY_TYPED) if strongly_typed else 0
    network = builder.create_network(flags)
    parser = trt.OnnxParser(network, logger)
    if not parser.parse_from_file(onnx_path):
        errs = "\n".join(str(parser.get_error(i)) for i in range(parser.num_errors))
        raise RuntimeError(f"TensorRT could not parse the ONNX model:\n{errs}")
    config = builder.create_builder_config()
    if not strongly_typed:
        config.set_flag(trt.BuilderFlag.FP16)
    config.set_memory_pool_limit(trt.MemoryPoolType.WORKSPACE, 4 << 30)
    t = time.time()
    serialized = builder.build_serialized_network(network, config)
    if serialized is None:
        raise RuntimeError("TensorRT engine build failed (see warnings above)")
    return serialized, time.time() - t, network.num_layers, strongly_typed


class TRTRunner:
    def __init__(self, serialized, trt):
        self.logger = trt.Logger(trt.Logger.WARNING)
        self.runtime = trt.Runtime(self.logger)
        self.engine = self.runtime.deserialize_cuda_engine(serialized)
        self.ctx = self.engine.create_execution_context()
        names = [self.engine.get_tensor_name(i) for i in range(self.engine.num_io_tensors)]
        self.inp = next(n for n in names if self.engine.get_tensor_mode(n) == trt.TensorIOMode.INPUT)
        self.outn = next(n for n in names if self.engine.get_tensor_mode(n) == trt.TensorIOMode.OUTPUT)
        dtypes = {trt.float32: torch.float32, trt.float16: torch.float16}
        self.out = torch.empty(tuple(self.ctx.get_tensor_shape(self.outn)),
                               dtype=dtypes[self.engine.get_tensor_dtype(self.outn)], device="cuda")

    def __call__(self, ids):
        self.ctx.set_tensor_address(self.inp, ids.data_ptr())
        self.ctx.set_tensor_address(self.outn, self.out.data_ptr())
        self.ctx.execute_async_v3(torch.cuda.current_stream().cuda_stream)
        return self.out


def main():
    p = argparse.ArgumentParser()
    p.add_argument("--model", default=DEFAULT_MODEL)
    p.add_argument("--seq-len", type=int, default=256)
    p.add_argument("--runs", type=int, default=50)
    p.add_argument("--build-dir", default="../trt_build")
    p.add_argument("--out", default="results")
    args = p.parse_args()

    import tensorrt as trt
    print(f"GPU: {torch.cuda.get_device_name()}   TensorRT {trt.__version__}   torch {torch.__version__}")
    os.makedirs(args.build_dir, exist_ok=True)
    os.makedirs(args.out, exist_ok=True)

    torch.manual_seed(0)
    ids = torch.randint(100, 20000, (1, args.seq_len), device="cuda", dtype=torch.int32)
    res = dict(model=args.model, gpu=torch.cuda.get_device_name(), seq_len=args.seq_len,
               trt_version=trt.__version__, runs=args.runs)

    # --- PyTorch baselines --------------------------------------------------------
    with torch.no_grad():
        sdpa = LastTokenLogits(load_model(args.model, "sdpa"))
        ref = sdpa(ids).clone()
        res["pytorch_ms_p50"], res["pytorch_ms_p99"] = time_cuda(lambda: sdpa(ids), runs=args.runs)
        print(f"  PyTorch eager (SDPA)  : {res['pytorch_ms_p50']:.2f} ms")
        try:
            comp = torch.compile(sdpa)
            res["compile_ms_p50"], res["compile_ms_p99"] = time_cuda(lambda: comp(ids), runs=args.runs)
            print(f"  torch.compile         : {res['compile_ms_p50']:.2f} ms")
        except Exception as e:
            print(f"  torch.compile failed: {type(e).__name__}: {e}")
        del sdpa
        torch._dynamo.reset()
        torch.cuda.empty_cache()

        # --- ONNX export -> TensorRT engine ----------------------------------------
        onnx_path = os.path.join(args.build_dir, "model.onnx")
        src = LastTokenLogits(load_model(args.model, "eager"))   # plain ops export cleanly
        res["onnx_exporter"] = export_onnx(src, ids, onnx_path)
        del src
        torch.cuda.empty_cache()

    print("  building TensorRT engine (a few minutes on a T4)...")
    serialized, build_s, n_layers, strongly_typed = build_engine(onnx_path, trt)
    with open(os.path.join(args.build_dir, "model.engine"), "wb") as f:
        f.write(serialized)
    res.update(build_s=build_s, network_layers=n_layers, strongly_typed=strongly_typed)
    print(f"  built in {build_s:.0f}s ({n_layers} network layers, strongly typed={strongly_typed})")

    runner = TRTRunner(serialized, trt)
    out = runner(ids).float().clone()
    torch.cuda.synchronize()
    res["trt_ms_p50"], res["trt_ms_p99"] = time_cuda(lambda: runner(ids), runs=args.runs)
    print(f"  TensorRT              : {res['trt_ms_p50']:.2f} ms")

    # --- correctness -----------------------------------------------------------------
    res["top1_match"] = bool(out.argmax(-1).eq(ref.argmax(-1)).all())
    top5_ref, top5_trt = set(ref.topk(5).indices[0].tolist()), set(out.topk(5).indices[0].tolist())
    res["top5_overlap"] = len(top5_ref & top5_trt)
    res["max_abs_logit_diff"] = (out - ref).abs().max().item()
    res["cosine_sim"] = torch.nn.functional.cosine_similarity(out, ref).item()
    res["speedup_vs_pytorch"] = res["pytorch_ms_p50"] / res["trt_ms_p50"]
    if "compile_ms_p50" in res:
        res["speedup_vs_compile"] = res["compile_ms_p50"] / res["trt_ms_p50"]

    print(f"  top-1 token match: {res['top1_match']}   top-5 overlap: {res['top5_overlap']}/5   "
          f"cosine sim: {res['cosine_sim']:.5f}   max |diff|: {res['max_abs_logit_diff']:.3f}")
    print(f"  TensorRT speedup: {res['speedup_vs_pytorch']:.2f}x vs PyTorch"
          + (f", {res['speedup_vs_compile']:.2f}x vs torch.compile" if "speedup_vs_compile" in res else ""))
    with open(os.path.join(args.out, "trt_prefill.json"), "w") as f:
        json.dump(res, f, indent=2)
    print(f"saved {args.out}/trt_prefill.json")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile perplexity.py
"""Accuracy cost of each optimization: WikiText-2 perplexity.

    python perplexity.py --variants sdpa_fp16 int8 nf4

Non-overlapping 1024-token windows over the first --max-tokens of the test set.
Kernel/compile variants should match FP16 almost exactly; quantized ones show the tradeoff.
"""
import argparse
import json
import math
import os
import subprocess
import sys
import time

import torch
from datasets import load_dataset

from common import DEFAULT_MODEL, free, load


CACHE = os.path.expanduser("~/.cache/wikitext2_test.txt")


def wikitext_text():
    """Download the WikiText-2 test split once (with retries, the Hub rate-limits
    anonymous requests) and cache it as plain text for every variant's process."""
    if os.path.exists(CACHE):
        with open(CACHE) as f:
            return f.read()
    for attempt in range(6):
        try:
            ds = load_dataset("Salesforce/wikitext", "wikitext-2-raw-v1", split="test")
            break
        except Exception as e:
            if attempt == 5:
                raise
            wait = 5 * 2 ** attempt
            print(f"  dataset download failed ({type(e).__name__}); retrying in {wait}s", flush=True)
            time.sleep(wait)
    text = "\n\n".join(ds["text"])
    os.makedirs(os.path.dirname(CACHE), exist_ok=True)
    with open(CACHE, "w") as f:
        f.write(text)
    return text


@torch.no_grad()
def perplexity(model, tok, max_tokens, window):
    ids = tok(wikitext_text(), return_tensors="pt").input_ids[:, :max_tokens].cuda()
    nll, count = 0.0, 0
    for i in range(0, ids.shape[1] - 1, window):
        chunk = ids[:, i: i + window]
        if chunk.shape[1] < 2:
            break
        loss = model(chunk, labels=chunk).loss
        nll += loss.item() * (chunk.shape[1] - 1)
        count += chunk.shape[1] - 1
    return math.exp(nll / count)


def main():
    p = argparse.ArgumentParser()
    p.add_argument("--variants", nargs="+", default=["sdpa_fp16", "triton_rmsnorm", "int8", "nf4"])
    p.add_argument("--model", default=DEFAULT_MODEL)
    p.add_argument("--max-tokens", type=int, default=40_000)
    p.add_argument("--window", type=int, default=1024)
    p.add_argument("--out", default="results")
    args = p.parse_args()

    os.makedirs(args.out, exist_ok=True)
    if len(args.variants) > 1:     # one process per variant, same reason as bench.py
        wikitext_text()            # download once up front
        for v in args.variants:
            cmd = [sys.executable, __file__, "--variants", v, "--model", args.model,
                   "--max-tokens", str(args.max_tokens), "--window", str(args.window),
                   "--out", args.out]
            if subprocess.run(cmd).returncode != 0:
                print(f"  !! {v} failed", flush=True)
        return

    for v in args.variants:
        tok, model = load(v, args.model)
        ppl = perplexity(model, tok, args.max_tokens, args.window)
        print(f"{v:>15}: perplexity {ppl:.3f}")
        with open(os.path.join(args.out, f"ppl_{v}.json"), "w") as f:
            json.dump({"variant": v, "perplexity": ppl, "tokens": args.max_tokens}, f, indent=2)
        free(model)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile profile_decode.py
"""Where does decode time go? Profiles 32 decode steps and prints the top GPU kernels.

    python profile_decode.py --variant sdpa_fp16

Also writes a Chrome trace (open at chrome://tracing or ui.perfetto.dev) so you can see
the launch gaps between tiny kernels — the thing CUDA graphs (compile variant) remove.
"""
import argparse
import os

import torch
from torch.profiler import ProfilerActivity, profile

from common import DEFAULT_MODEL, VARIANTS, load


def main():
    p = argparse.ArgumentParser()
    p.add_argument("--variant", default="sdpa_fp16", choices=list(VARIANTS))
    p.add_argument("--model", default=DEFAULT_MODEL)
    p.add_argument("--out", default="results")
    args = p.parse_args()

    tok, model = load(args.variant, args.model)
    ids = tok("Explain how a GPU executes a matrix multiply.", return_tensors="pt").input_ids.cuda()
    gen = dict(max_new_tokens=32, min_new_tokens=32, do_sample=False,
               pad_token_id=model.config.eos_token_id)
    for _ in range(3 if args.variant == "compile" else 1):
        model.generate(ids, **gen)            # warmup
    torch.cuda.synchronize()

    with profile(activities=[ProfilerActivity.CPU, ProfilerActivity.CUDA]) as prof:
        model.generate(ids, **gen)
        torch.cuda.synchronize()

    events = prof.key_averages()
    for key in ("cuda_time_total", "device_time_total"):
        try:
            print(events.table(sort_by=key, row_limit=15))
            break
        except Exception:
            continue

    os.makedirs(args.out, exist_ok=True)
    path = os.path.join(args.out, f"trace_{args.variant}.json")
    prof.export_chrome_trace(path)
    print(f"trace -> {path}")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile report.py
"""Collect results/*.json into tables, headline findings and a chart.

Writes results/summary.md and results/summary.png, and fills the README section
between the RESULTS markers so the repo always shows the numbers from the last run.
"""
import glob
import json
import math
import os
import re

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

ORDER = ["eager_fp16", "sdpa_fp16", "triton_rmsnorm", "compile", "int8", "nf4"]
SAME_MATH_AS_FP16 = {"eager_fp16", "compile"}      # identical weights/precision to sdpa_fp16


def jload(path):
    with open(path) as f:
        return json.load(f)


def fmt(x, spec):
    return "—" if x is None or (isinstance(x, float) and math.isnan(x)) else format(x, spec)


def build(out):
    bench = {d["variant"]: d for d in map(jload, glob.glob(os.path.join(out, "bench_*.json")))}
    ppl = {d["variant"]: d["perplexity"] for d in map(jload, glob.glob(os.path.join(out, "ppl_*.json")))}
    kpath, tpath = os.path.join(out, "kernel_rmsnorm.json"), os.path.join(out, "trt_prefill.json")
    kern = jload(kpath) if os.path.exists(kpath) else None
    trt = jload(tpath) if os.path.exists(tpath) else None
    if not bench:
        raise SystemExit("no bench_*.json found — run bench.py first")

    names = [v for v in ORDER if v in bench]
    base = bench.get("eager_fp16") or bench[names[0]]
    for v in SAME_MATH_AS_FP16:                       # these compute the same thing as sdpa_fp16
        if v not in ppl and "sdpa_fp16" in ppl:
            ppl[v] = ppl["sdpa_fp16"]

    # ---------------------------------------------------------------- headline findings
    find = []
    if "compile" in bench:
        c = bench["compile"]
        find.append(f"**torch.compile + static KV cache: {c['decode_tok_s'] / base['decode_tok_s']:.1f}x decode throughput** "
                    f"({base['decode_tok_s']:.1f} → {c['decode_tok_s']:.1f} tok/s). Per-token latency fell from "
                    f"{base['itl_ms_p50']:.1f} ms to {c['itl_ms_p50']:.1f} ms, and p99 from "
                    f"{base['itl_ms_p99']:.0f} ms to {c['itl_ms_p99']:.1f} ms.")
    if trt:
        line = (f"**TensorRT engine: {trt['speedup_vs_pytorch']:.1f}x faster prefill than PyTorch** "
                f"({trt['pytorch_ms_p50']:.1f} → {trt['trt_ms_p50']:.1f} ms for a {trt['seq_len']}-token prompt")
        if "speedup_vs_compile" in trt:
            line += f", {trt['speedup_vs_compile']:.2f}x vs torch.compile"
        line += (f"), with matching outputs (top-1 token match: {'yes' if trt['top1_match'] else 'no'}, "
                 f"cosine similarity {trt['cosine_sim']:.4f}).")
        find.append(line)
    if kern:
        big = [r for r in kern["rows"] if r["M"] >= 512]
        sp = sorted(r["speedup"] for r in big)[len(big) // 2]
        vs_comp = sorted(r["compile_us"] / r["triton_us"] for r in big)[len(big) // 2]
        peak = max(r["pct_peak"] for r in kern["rows"])
        find.append(f"**Custom Triton fused add+RMSNorm kernel: {sp:.1f}x faster than eager PyTorch** "
                    f"(median over prefill-sized shapes), {vs_comp:.2f}x vs torch.compile, reaching "
                    f"{peak:.0f}% of the GPU's peak memory bandwidth.")
    if "nf4" in bench and "sdpa_fp16" in ppl and "nf4" in ppl:
        n, f16 = bench["nf4"], bench["sdpa_fp16"]
        find.append(f"**4-bit NF4 cut peak GPU memory {100 * (1 - n['peak_mem_mb'] / f16['peak_mem_mb']):.0f}%** "
                    f"({f16['peak_mem_mb']:.0f} → {n['peak_mem_mb']:.0f} MB) at a "
                    f"{100 * (ppl['nf4'] / ppl['sdpa_fp16'] - 1):.0f}% perplexity cost "
                    f"({ppl['sdpa_fp16']:.2f} → {ppl['nf4']:.2f}).")
    if "int8" in bench and "int8" in ppl and "sdpa_fp16" in ppl:
        i8 = bench["int8"]
        find.append(f"**INT8 kept accuracy** (perplexity {ppl['int8']:.2f} vs {ppl['sdpa_fp16']:.2f}) "
                    f"**but ran {base['decode_tok_s'] / i8['decode_tok_s']:.1f}x slower** at batch 1: "
                    f"bitsandbytes LLM.int8 dequantizes weights on every matmul, so at batch 1 that overhead dominates.")

    # ---------------------------------------------------------------- tables
    L = [f"Model `{base['model']}` on **{base['gpu']}**. Prompt {base['prompt_len']} tokens, "
         f"generate {base['new_tokens']} tokens, batch 1, greedy, {base['runs']} timed runs per variant.", "",
         "### Key findings", ""] + [f"- {x}" for x in find] + [
         "", "### End-to-end generation", "",
         "| Variant | Decode tok/s | Speedup | TTFT p50 (ms) | Per-token p50 (ms) | Per-token p99 (ms) | Peak memory (MB) | WikiText-2 perplexity |",
         "|---|---:|---:|---:|---:|---:|---:|---:|"]
    for v in names:
        b = bench[v]
        L.append(f"| `{v}` | {b['decode_tok_s']:.1f} | {b['decode_tok_s'] / base['decode_tok_s']:.2f}x | "
                 f"{b['ttft_ms_p50']:.1f} | {b['itl_ms_p50']:.2f} | {b['itl_ms_p99']:.2f} | "
                 f"{b['peak_mem_mb']:.0f} | {fmt(ppl.get(v), '.3f')} |")
    L += ["", "![Benchmark summary](results/summary.png)"]

    if trt:
        L += ["", f"### TensorRT prefill ({trt['seq_len']}-token prompt, TensorRT {trt['trt_version']})", "",
              "| Backend | p50 (ms) | p99 (ms) | Speedup |", "|---|---:|---:|---:|",
              f"| PyTorch eager (SDPA) | {trt['pytorch_ms_p50']:.2f} | {trt['pytorch_ms_p99']:.2f} | 1.00x |"]
        if "compile_ms_p50" in trt:
            L.append(f"| torch.compile | {trt['compile_ms_p50']:.2f} | {trt['compile_ms_p99']:.2f} | "
                     f"{trt['pytorch_ms_p50'] / trt['compile_ms_p50']:.2f}x |")
        L += [f"| TensorRT engine | {trt['trt_ms_p50']:.2f} | {trt['trt_ms_p99']:.2f} | {trt['speedup_vs_pytorch']:.2f}x |",
              "", f"Output check vs PyTorch: top-1 token match **{'yes' if trt['top1_match'] else 'no'}**, "
              f"top-5 overlap {trt['top5_overlap']}/5, cosine similarity {trt['cosine_sim']:.5f}. "
              f"Engine build: {trt['build_s']:.0f} s, {trt['network_layers']} network layers."]

    if kern:
        L += ["", f"### Triton kernel microbenchmark ({kern['gpu']}, peak {kern['peak_gbs']} GB/s)", "",
              "M = rows (1 = one decode token, 4096 = long prefill), N = hidden size.", "",
              "| M | N | Eager PyTorch (µs) | torch.compile (µs) | Triton (µs) | vs eager | vs compile | GB/s | % of peak |",
              "|---:|---:|---:|---:|---:|---:|---:|---:|---:|"]
        for r in kern["rows"]:
            L.append(f"| {r['M']} | {r['N']} | {r['eager_us']:.1f} | {r['compile_us']:.1f} | {r['triton_us']:.1f} | "
                     f"{r['speedup']:.2f}x | {r['compile_us'] / r['triton_us']:.2f}x | {r['gbs']:.0f} | {fmt(r['pct_peak'], '.0f')}% |")

    # ---------------------------------------------------------------- chart
    fig, ax = plt.subplots(1, 3, figsize=(15, 4.2))
    metrics = [("decode_tok_s", "Decode tokens/sec (higher is better)"),
               ("itl_ms_p99", "Per-token latency p99, ms (lower is better)"),
               ("peak_mem_mb", "Peak GPU memory, MB (lower is better)")]
    for a, (key, title) in zip(ax, metrics):
        vals = [bench[v][key] for v in names]
        bars = a.bar(names, vals, color=["#76B900" if v == "compile" else "#4C72B0" for v in names])
        a.bar_label(bars, fmt="%.0f" if key != "itl_ms_p99" else "%.1f", fontsize=8)
        a.set_title(title, fontsize=10)
        a.tick_params(axis="x", rotation=35)
        a.grid(axis="y", alpha=0.3)
        a.spines[["top", "right"]].set_visible(False)
    fig.suptitle(f"{base['model']} on {base['gpu']}, batch 1")
    fig.tight_layout()
    fig.savefig(os.path.join(out, "summary.png"), dpi=140)
    return "\n".join(L), find


def main(out="results", readme="README.md"):
    md, find = build(out)
    with open(os.path.join(out, "summary.md"), "w") as f:
        f.write(md + "\n")
    if os.path.exists(readme):
        with open(readme) as f:
            text = f.read()
        new = re.sub(r"<!-- RESULTS:START -->.*<!-- RESULTS:END -->",
                     lambda _: f"<!-- RESULTS:START -->\n{md}\n<!-- RESULTS:END -->", text, flags=re.S)
        with open(readme, "w") as f:
            f.write(new)
    print(md)
    print(f"\nsaved {out}/summary.md, {out}/summary.png and updated {readme}")


if __name__ == "__main__":
    main()


In [ ]:
!pip install -q -r requirements.txt

## 1. Triton kernel: correctness + microbenchmark

In [ ]:
!python -m kernels.rmsnorm_triton

## 2. End-to-end latency, every variant

In [ ]:
!python bench.py

## 3. TensorRT engine (ONNX export → TensorRT build, a few minutes)

In [ ]:
!pip install -q tensorrt
!python trt_prefill.py

## 4. Accuracy cost (perplexity)

In [ ]:
!python perplexity.py

## 5. Profile decode: eager vs compiled

In [ ]:
!python profile_decode.py --variant sdpa_fp16 | tail -25
!python profile_decode.py --variant compile | tail -25

## 6. Results, chart and README

In [ ]:
!python report.py > /dev/null
from IPython.display import Image, Markdown, display
display(Markdown(open('results/summary.md').read().replace('![Benchmark summary](results/summary.png)', '')))
display(Image('results/summary.png'))

## 7. Download the finished project
Send this zip back to Claude to publish on GitHub.

In [ ]:
%cd /content
!zip -qr llm-inference-opt.zip llm-inference-opt -x 'llm-inference-opt/results/trace_*' -x '*__pycache__*'
!ls -lh llm-inference-opt.zip
from google.colab import files
files.download('llm-inference-opt.zip')